# Exploración de Bronze
Antes de escribir el transformer, entendemos qué hay dentro de los JSONs que acabamos de descargar.

**Objetivo:** confirmar estructura real de los datos para escribir el transformer sin sorpresas.

In [4]:
from pathlib import Path
import json
import pandas as pd
import glob
from pprint import pprint

# Localizamos la raíz aunque el notebook se ejecute desde explorations/
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'data' / 'bronze').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / 'data' / 'bronze').is_dir():
    raise FileNotFoundError('No se encontró la carpeta data/bronze desde el directorio actual')

DATA_DIR = PROJECT_ROOT / 'data'
football_files = glob.glob(str(DATA_DIR / 'bronze' / 'football' / '*.json'))
weather_files = glob.glob(str(DATA_DIR / 'bronze' / 'weather' / '*.json'))

print('Raíz del proyecto:', PROJECT_ROOT)
print('Archivos football:', football_files)
print('Archivos weather: ', len(weather_files), 'equipos')

Raíz del proyecto: c:\Users\anaos\Desktop\proyectos\sports-pipeline
Archivos football: ['c:\\Users\\anaos\\Desktop\\proyectos\\sports-pipeline\\data\\bronze\\football\\20260820_matches_PD_2023.json', 'c:\\Users\\anaos\\Desktop\\proyectos\\sports-pipeline\\data\\bronze\\football\\20260820_standings_PD_2023.json', 'c:\\Users\\anaos\\Desktop\\proyectos\\sports-pipeline\\data\\bronze\\football\\20260902_matches_PD_2023.json', 'c:\\Users\\anaos\\Desktop\\proyectos\\sports-pipeline\\data\\bronze\\football\\20260902_standings_PD_2023.json', 'c:\\Users\\anaos\\Desktop\\proyectos\\sports-pipeline\\data\\bronze\\football\\20260903_matches_PD_2023.json', 'c:\\Users\\anaos\\Desktop\\proyectos\\sports-pipeline\\data\\bronze\\football\\20260903_standings_PD_2023.json', 'c:\\Users\\anaos\\Desktop\\proyectos\\sports-pipeline\\data\\bronze\\football\\20260904_matches_PD_2023.json', 'c:\\Users\\anaos\\Desktop\\proyectos\\sports-pipeline\\data\\bronze\\football\\20260904_standings_PD_2023.json']
Archivos

---
## 1. Partidos (football-data.org)

In [8]:
matches_file = [f for f in football_files if 'matches' in f][0]
print('Leyendo:', matches_file)

with open(matches_file, encoding='utf-8') as f:
    matches_raw = json.load(f)

print('Claves del JSON:', list(matches_raw.keys()))
print('Total partidos:', matches_raw['resultSet']['count'])

Leyendo: c:\Users\anaos\Desktop\proyectos\sports-pipeline\data\bronze\football\20260820_matches_PD_2023.json
Claves del JSON: ['filters', 'resultSet', 'competition', 'matches']
Total partidos: 380


In [7]:
# Estructura completa de un partido
pprint(matches_raw['matches'][0])

{'area': {'code': 'ESP',
          'flag': 'https://crests.football-data.org/760.svg',
          'id': 2224,
          'name': 'Spain'},
 'awayTeam': {'crest': 'https://crests.football-data.org/87.png',
              'id': 87,
              'name': 'Rayo Vallecano de Madrid',
              'shortName': 'Rayo Vallecano',
              'tla': 'RAY'},
 'competition': {'code': 'PD',
                 'emblem': 'https://crests.football-data.org/laliga.png',
                 'id': 2014,
                 'name': 'Primera Division',
                 'type': 'LEAGUE'},
 'group': None,
 'homeTeam': {'crest': 'https://crests.football-data.org/267.png',
              'id': 267,
              'name': 'UD Almería',
              'shortName': 'Almería',
              'tla': 'ALM'},
 'id': 438482,
 'lastUpdated': '2023-10-09T15:20:25Z',
 'matchday': 1,
 'odds': {'msg': 'Activate Odds-Package in User-Panel to retrieve odds.'},
 'referees': [{'id': 80747,
               'name': 'Javier Alberola Rojas',
 

In [11]:
# PUNTO CLAVE 1: nombres de equipos según la API
equipos_api = set()
for m in matches_raw['matches']:
    equipos_api.add(m['homeTeam']['name'])
    equipos_api.add(m['awayTeam']['name'])

print('Equipos según la API:')
for e in sorted(equipos_api):
    print(' ', e)

Equipos según la API:
  Athletic Club
  CA Osasuna
  Club Atlético de Madrid
  Cádiz CF
  Deportivo Alavés
  FC Barcelona
  Getafe CF
  Girona FC
  Granada CF
  RC Celta de Vigo
  RCD Mallorca
  Rayo Vallecano de Madrid
  Real Betis Balompié
  Real Madrid CF
  Real Sociedad de Fútbol
  Sevilla FC
  UD Almería
  UD Las Palmas
  Valencia CF
  Villarreal CF


In [13]:
# Comparamos con stadiums.json — detectamos mismatches antes de escribir el transformer
with open(DATA_DIR / 'stadiums.json', encoding='utf-8') as f:
    stadiums = json.load(f)

equipos_stadiums = set(stadiums.keys())
sin_match = equipos_api - equipos_stadiums

print('Equipos de la API que NO están en stadiums.json:')
if sin_match:
    for e in sorted(sin_match):
        print(' ❌', e)
else:
    print(' ✅ Todos coinciden')

Equipos de la API que NO están en stadiums.json:
 ✅ Todos coinciden


In [14]:
# PUNTO CLAVE 2: valores del campo winner
winners = set(m['score'].get('winner') for m in matches_raw['matches'])
print('Valores posibles de score.winner:', winners)

sin_winner = [m for m in matches_raw['matches'] if m['score'].get('winner') is None]
print(f'Partidos sin winner: {len(sin_winner)}')
print('Status de esos partidos:', set(m['status'] for m in sin_winner))

Valores posibles de score.winner: {'AWAY_TEAM', 'DRAW', 'HOME_TEAM'}
Partidos sin winner: 0
Status de esos partidos: set()


In [15]:
# DataFrame general de partidos
rows = []
for m in matches_raw['matches']:
    rows.append({
        'match_id':   m['id'],
        'date':       m['utcDate'],
        'status':     m['status'],
        'home_team':  m['homeTeam']['name'],
        'away_team':  m['awayTeam']['name'],
        'home_score': m['score']['fullTime'].get('home'),
        'away_score': m['score']['fullTime'].get('away'),
        'winner':     m['score'].get('winner'),
        'matchday':   m.get('matchday'),
    })

df_matches = pd.DataFrame(rows)
df_matches['date'] = pd.to_datetime(df_matches['date'])

print('Shape:', df_matches.shape)
print('\nEstados:')
print(df_matches['status'].value_counts())
df_matches.head(10)

Shape: (380, 9)

Estados:
status
FINISHED    380
Name: count, dtype: int64


,match_id,date,status,home_team,away_team,home_score,away_score,winner,matchday
0,438482,2023-08-11 17:30:00+00:00,FINISHED,UD Almería,Rayo Vallecano de Madrid,0,2,AWAY_TEAM,1
1,438479,2023-08-11 20:00:00+00:00,FINISHED,Sevilla FC,Valencia CF,1,2,AWAY_TEAM,1
2,438481,2023-08-12 15:00:00+00:00,FINISHED,Real Sociedad de Fútbol,Girona FC,1,1,DRAW,1
3,438483,2023-08-12 17:30:00+00:00,FINISHED,UD Las Palmas,RCD Mallorca,1,1,DRAW,1
4,438474,2023-08-12 19:30:00+00:00,FINISHED,Athletic Club,Real Madrid CF,0,2,AWAY_TEAM,1
5,438476,2023-08-13 15:00:00+00:00,FINISHED,RC Celta de Vigo,CA Osasuna,0,2,AWAY_TEAM,1
6,438480,2023-08-13 17:30:00+00:00,FINISHED,Villarreal CF,Real Betis Balompié,1,2,AWAY_TEAM,1
7,438478,2023-08-13 19:30:00+00:00,FINISHED,Getafe CF,FC Barcelona,0,0,DRAW,1
8,438477,2023-08-14 17:30:00+00:00,FINISHED,Cádiz CF,Deportivo Alavés,1,0,HOME_TEAM,1
9,438475,2023-08-14 19:30:00+00:00,FINISHED,Club Atlético de Madrid,Granada CF,3,1,HOME_TEAM,1


In [16]:
# Nulos en columnas clave
df_matches[['match_id','date','home_team','away_team','home_score','away_score','winner']].isnull().sum()

match_id      0
date          0
home_team     0
away_team     0
home_score    0
away_score    0
winner        0
dtype: int64

---
## 2. Clima (Open-Meteo)

In [17]:
sample_file = weather_files[0]
print('Leyendo:', sample_file)

with open(sample_file, encoding='utf-8') as f:
    weather_raw = json.load(f)

print('Meta:', weather_raw.get('_meta'))
print('Días descargados:', len(weather_raw['daily']['time']))
pprint(weather_raw['daily'])

Leyendo: c:\Users\anaos\Desktop\proyectos\sports-pipeline\data\bronze\weather\20260820_athletic_club_weather.json
Meta: {'team': 'Athletic Club', 'stadium': 'Estadio de San Mamés', 'city': 'Bilbao'}
Días descargados: 335
{'precipitation_sum': [1.0,
                       0.6,
                       2.4,
                       9.3,
                       0.4,
                       1.8,
                       0.0,
                       0.0,
                       0.0,
                       0.0,
                       0.0,
                       2.2,
                       0.0,
                       0.5,
                       0.6,
                       3.3,
                       0.0,
                       0.0,
                       0.0,
                       0.0,
                       0.0,
                       0.0,
                       0.0,
                       0.0,
                       1.6,
                       14.3,
                       10.2,
                     

In [18]:
# Cargamos todos los equipos
all_dfs = []
for wf in weather_files:
    with open(wf, encoding='utf-8') as f:
        wd = json.load(f)
    daily = wd['daily']
    df = pd.DataFrame({
        'date':          pd.to_datetime(daily['time']),
        'team':          wd['_meta']['team'],
        'city':          wd['_meta']['city'],
        'temp_max':      daily['temperature_2m_max'],
        'temp_min':      daily['temperature_2m_min'],
        'temp_avg':      [(mx+mn)/2 for mx,mn in zip(daily['temperature_2m_max'], daily['temperature_2m_min'])],
        'precipitation': daily['precipitation_sum'],
        'wind_max':      daily['windspeed_10m_max'],
        'weather_code':  daily['weathercode'],
    })
    all_dfs.append(df)

df_weather_all = pd.concat(all_dfs, ignore_index=True)
print('Shape total weather:', df_weather_all.shape)
print('Equipos:', df_weather_all['team'].nunique())
df_weather_all.sample(5)

Shape total weather: (6700, 9)
Equipos: 20


,date,team,city,temp_max,temp_min,temp_avg,precipitation,wind_max,weather_code
5492,2023-12-11,UD Almería,Almería,20.4,13.4,16.90,0.0,9.1,3
2336,2024-06-22,Getafe CF,Getafe,31.6,15.0,23.30,0.0,18.4,3
3238,2024-03-11,Rayo Vallecano de Madrid,Madrid,13.2,5.5,9.35,0.0,21.6,3
6203,2024-01-21,Valencia CF,Valencia,15.9,6.1,11.00,0.0,14.7,3
1123,2023-11-27,Cádiz CF,Cádiz,16.8,9.8,13.30,0.0,14.7,3


---
## 3. Join de prueba: partidos + clima

In [19]:
df_finished = df_matches[df_matches['status'] == 'FINISHED'].copy()
print('Partidos terminados:', len(df_finished))

df_finished['date_only'] = df_finished['date'].dt.date
df_weather_all['date_only'] = df_weather_all['date'].dt.date

df_weather_join = df_weather_all.rename(columns={'team': 'home_team'}).drop(columns='date')

df_merged = df_finished.merge(
    df_weather_join,
    on=['home_team', 'date_only'],
    how='left'
)

print('Shape merged:', df_merged.shape)
print('\nNulos en columnas de clima:')
print(df_merged[['temp_avg', 'precipitation', 'wind_max']].isnull().sum())

Partidos terminados: 380
Shape merged: (380, 17)

Nulos en columnas de clima:
temp_avg         0
precipitation    0
wind_max         0
dtype: int64


In [20]:
# Si hay NaN, ¿de qué equipos vienen?
sin_clima = df_merged[df_merged['temp_avg'].isna()]
if len(sin_clima) > 0:
    print('Partidos sin clima por equipo local:')
    print(sin_clima['home_team'].value_counts())
    print('\nFechas sin clima:')
    print(sin_clima['date_only'].unique())
else:
    print('✅ Todos los partidos tienen datos de clima')

✅ Todos los partidos tienen datos de clima


In [21]:
# Vista previa del resultado final
cols = ['match_id', 'date_only', 'home_team', 'away_team', 'winner',
        'home_score', 'away_score', 'temp_avg', 'precipitation', 'wind_max']
df_merged[cols].head(10)

,match_id,date_only,home_team,away_team,winner,home_score,away_score,temp_avg,precipitation,wind_max
0,438482,2023-08-11,UD Almería,Rayo Vallecano de Madrid,AWAY_TEAM,0,2,31.25,0.0,22.4
1,438479,2023-08-11,Sevilla FC,Valencia CF,AWAY_TEAM,1,2,34.60,0.0,22.7
2,438481,2023-08-12,Real Sociedad de Fútbol,Girona FC,DRAW,1,1,21.25,2.5,14.8
3,438483,2023-08-12,UD Las Palmas,RCD Mallorca,DRAW,1,1,28.60,0.0,18.5
4,438474,2023-08-12,Athletic Club,Real Madrid CF,AWAY_TEAM,0,2,21.25,2.2,17.3
5,438476,2023-08-13,RC Celta de Vigo,CA Osasuna,AWAY_TEAM,0,2,20.40,0.0,21.9
6,438480,2023-08-13,Villarreal CF,Real Betis Balompié,AWAY_TEAM,1,2,26.40,0.0,14.6
7,438478,2023-08-13,Getafe CF,FC Barcelona,DRAW,0,0,30.60,0.0,21.4
8,438477,2023-08-14,Cádiz CF,Deportivo Alavés,HOME_TEAM,1,0,24.70,0.0,13.5
9,438475,2023-08-14,Club Atlético de Madrid,Granada CF,HOME_TEAM,3,1,28.80,0.0,15.1


In [5]:
import pandas as pd
import glob

files = sorted(glob.glob(str(DATA_DIR / "silver" / "*.parquet")))
print("Archivos silver:", files)

if not files:
    raise FileNotFoundError(f"No se encontraron archivos Parquet en {DATA_DIR / 'silver'}")

df = pd.read_parquet(files[-1])

# ¿Cuántas filas por match_id?
duplicados = df.groupby("match_id").size()
print(duplicados.value_counts())

Archivos silver: ['c:\\Users\\anaos\\Desktop\\proyectos\\sports-pipeline\\data\\silver\\20260821_matches_weather.parquet', 'c:\\Users\\anaos\\Desktop\\proyectos\\sports-pipeline\\data\\silver\\20260902_matches_weather.parquet', 'c:\\Users\\anaos\\Desktop\\proyectos\\sports-pipeline\\data\\silver\\20260903_matches_weather.parquet', 'c:\\Users\\anaos\\Desktop\\proyectos\\sports-pipeline\\data\\silver\\20260904_matches_weather.parquet']
3    380
Name: count, dtype: int64
